In [2]:
!pip install torch torchvision numpy pybullet matplotlib

  Using cached pybullet-3.2.7.tar.gz (80.5 MB)
  Preparing metadata (setup.py) ... done
  Created wheel for pybullet: filename=pybullet-3.2.7-cp313-cp313-linux_x86_64.whl size=99120539 sha256=1b454cfbb858df4e458433dd5b66f82d62efbd59535e162e23df5982b07935df
  Stored in directory: /root/.cache/pip/wheels/8a/f0/6a/3f626b4ee04f8ec61e40021b095adf5359f9bd426065a368ac
Successfully built pybullet


In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [4]:
from torch.utils.data import Dataset, DataLoader

class SyntheticTactileDataset(Dataset):
    def __init__(self, num_samples=1000, latent_dim=128):
        self.x = torch.randn(num_samples, 3)              # 3D spatial points (x, y, z)
        self.latents = torch.randn(num_samples, latent_dim) # Shape latent code z
        self.forces = torch.abs(torch.randn(num_samples, 1)) * 5.0 # Normal force [0 to 5N]
        self.stiffness = torch.full((num_samples, 1), 500.0)       # Material stiffness k (500 N/m)

        self.s_gt = torch.zeros(num_samples, 1)            # Points on surface
        self.tactile_depth = self.forces / self.stiffness  # Delta D = F / k

    def __len__(self):
        return len(self.x)

    def __getitem__(self, idx):
        return self.x[idx], self.latents[idx], self.forces[idx], self.stiffness[idx], self.s_gt[idx], self.tactile_depth[idx]

# Creating dataset and dataloader
dataset = SyntheticTactileDataset(num_samples=1000)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)
print("Dataset created with", len(dataset), "samples.")

Dataset created with 1000 samples.


In [5]:
class DeformableTouchSDF(nn.Module):
    def __init__(self, latent_dim=128):
        super().__init__()

        # Branch 1: Base Shape MLP (Predicts un-dented nominal shape S0)
        self.base_mlp = nn.Sequential(
            nn.Linear(3 + latent_dim, 256),
            nn.ReLU(),
            nn.Linear(256, 256),
            nn.ReLU(),
            nn.Linear(256, 1)
        )

        # Branch 2: Offset Subnet (Predicts dent depth using Force and Compliance)
        self.offset_net = nn.Sequential(
            nn.Linear(2, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
            nn.Softplus() # Keeps indentation depth strictly positive
        )

    def forward(self, x, latent_code, force, stiffness):
        # Calculate material compliance m = 1 / k
        compliance = 1.0 / (stiffness + 1e-6)

        # 1. Predict nominal shape
        base_input = torch.cat([x, latent_code], dim=-1)
        s_0 = self.base_mlp(base_input)

        # 2. Predict surface indentation displacement offset
        offset_input = torch.cat([force, compliance], dim=-1)
        delta_d = self.offset_net(offset_input)

        # 3. Deformed SDF = Nominal Shape - Indent Dent
        s_deformed = s_0 - delta_d
        return s_deformed, s_0, delta_d

In [6]:
class DeformableSDFLoss(nn.Module):
    def __init__(self, lambda_contact=0.1, lambda_physics=0.01):
        super().__init__()
        self.lambda_contact = lambda_contact
        self.lambda_physics = lambda_physics
        self.l1_loss = nn.L1Loss()
        self.mse_loss = nn.MSELoss()

    def forward(self, s_deformed_pred, s_gt, delta_d_pred, force, stiffness, tactile_depth_gt):
        # 1. Geometric SDF reconstruction loss
        l_sdf = self.l1_loss(s_deformed_pred, s_gt)

        # 2. Optical tactile sensor contact depth loss
        l_contact = self.mse_loss(delta_d_pred, tactile_depth_gt)

        # 3. Physical elasticity prior (Hooke's Law: Delta D = Force / Stiffness)
        expected_displacement = force / (stiffness + 1e-6)
        l_physics = self.mse_loss(delta_d_pred, expected_displacement)

        # Total combined loss
        total_loss = l_sdf + (self.lambda_contact * l_contact) + (self.lambda_physics * l_physics)
        return total_loss

In [7]:
# Initializing model, loss function, and optimizer
model = DeformableTouchSDF(latent_dim=128).to(device)
criterion = DeformableSDFLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

num_epochs = 100
model.train()

print("Starting training loop...")
for epoch in range(num_epochs):
    epoch_loss = 0.0

    for batch in dataloader:
        x, latent_code, force, stiffness, s_gt, tactile_depth = [b.to(device) for b in batch]
        optimizer.zero_grad()
        s_deformed, s_0, delta_d = model(x, latent_code, force, stiffness)
        loss = criterion(s_deformed, s_gt, delta_d, force, stiffness, tactile_depth)

        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    print(f"Epoch [{epoch+1}/{num_epochs}] | Average Loss: {epoch_loss / len(dataloader):.6f}")

print("Training complete!")

Starting training loop...
Epoch [1/100] | Average Loss: 0.209819
Epoch [2/100] | Average Loss: 0.170865
Epoch [3/100] | Average Loss: 0.155676
Epoch [4/100] | Average Loss: 0.139850
Epoch [5/100] | Average Loss: 0.125370
Epoch [6/100] | Average Loss: 0.109281
Epoch [7/100] | Average Loss: 0.097534
Epoch [8/100] | Average Loss: 0.083900
Epoch [9/100] | Average Loss: 0.072949
Epoch [10/100] | Average Loss: 0.065669
Epoch [11/100] | Average Loss: 0.057794
Epoch [12/100] | Average Loss: 0.052970
Epoch [13/100] | Average Loss: 0.048512
Epoch [14/100] | Average Loss: 0.045470
Epoch [15/100] | Average Loss: 0.043950
Epoch [16/100] | Average Loss: 0.043961
Epoch [17/100] | Average Loss: 0.040330
Epoch [18/100] | Average Loss: 0.037431
Epoch [19/100] | Average Loss: 0.035609
Epoch [20/100] | Average Loss: 0.034556
Epoch [21/100] | Average Loss: 0.033665
Epoch [22/100] | Average Loss: 0.032745
Epoch [23/100] | Average Loss: 0.032217
Epoch [24/100] | Average Loss: 0.031491
Epoch [25/100] | Averag

In [8]:
model.eval()

with torch.no_grad():
    # Generating a small test batch
    test_x = torch.randn(50, 3).to(device)
    test_latent = torch.randn(50, 128).to(device)
    test_force = torch.ones(50, 1).to(device) * 2.0
    test_stiffness = torch.full((50, 1), 500.0).to(device)

    # Running forward pass to get un-dented nominal shape (S0)
    _, s_0_pred, _ = model(test_x, test_latent, test_force, test_stiffness)

    # Computing Mean Absolute Error against ground-truth as a proxy error metric
    evaluation_error = torch.mean(torch.abs(s_0_pred)).item()
    print(f"Model Evaluation Complete | Reconstructed Shape Error: {evaluation_error:.4f}")

Model Evaluation Complete | Reconstructed Shape Error: 0.2281
